# 10: ISIC 2019 Bronze ingestion

Indexes every image in the archives (checksum and location, no bytes) and loads the metadata as published. Run after `05_setup_tables_and_folders`, with the archives uploaded.

Produces `bronze.isic_2019_image_index`, `bronze.isic_2019_source_metadata` and a row in `bronze.ingestion_runs`.

In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

from data_platform.files import check_archives_exist, stage_archives_and_extract_metadata
from data_platform.dataset_layout import load_dataset
from data_platform.spark_io import (
    load_combined_metadata,
    print_bronze_outputs,
    write_image_index_table,
    write_ingestion_run,
    write_source_metadata,
)

import tempfile


dataset = load_dataset(CONFIG_ROOT, "isic_2019")
DATASET_CONFIG = dataset["config"]
ARCHIVES = dataset["archives"]
landing_paths, bronze_paths, bronze_tables = dataset["landing_paths"], dataset["bronze_paths"], dataset["bronze_tables"]
LOCAL_ARCHIVE_STAGE_ROOT = Path(tempfile.gettempdir()) / DATASET_CONFIG["dataset_key"] / "archive_stage"
OVERWRITE_LOCAL_STAGE = False

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive landing path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image index table: {bronze_tables['image_index']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Check the archives are in landing

In [ ]:
check_archives_exist(ARCHIVES)


## Stage archives locally and extract the metadata CSV

Each archive is copied once to local disk, so no step makes one cloud request per image.

In [ ]:
stage_archives_and_extract_metadata(ARCHIVES, LOCAL_ARCHIVE_STAGE_ROOT, OVERWRITE_LOCAL_STAGE)

display(dbutils.fs.ls(bronze_paths["metadata"]))


## Write the image index

Checksums and archive locations only; image bytes are read and discarded ([ADR 004](../../docs/decisions/004-stream-archives-no-blob-storage.md)).

In [ ]:
image_manifest_df = write_image_index_table(spark, ARCHIVES, bronze_tables["image_index"])


## Write the source metadata and log the run

The metadata schema below is ISIC 2019's own; it's the one dataset-specific step in Bronze.

In [ ]:
raw_metadata_df = load_combined_metadata(spark, ARCHIVES)

ingestion_run_id = f"bronze-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)
records_seen = raw_metadata_df.count()  # before rows without a matching image are dropped

# ISIC 2019's actual Bronze source metadata schema — dataset-specific by design, unlike
# everything else in this notebook (see docs/data_contract.md).
source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", F.col("isic_id").cast("string"))
    .join(image_manifest_df, on=["image_id", "source_split"], how="left")
    .select(
        F.col("image_id"),
        F.col("source_split"),
        F.col("source_uri"),
        F.col("attribution"),
        F.col("copyright_license"),
        F.col("age_approx"),
        F.col("anatom_site_1"),
        F.col("anatom_site_2"),
        F.col("anatom_site_3"),
        F.col("anatom_site_4"),
        F.col("anatom_site_5"),
        F.col("anatom_site_special"),
        F.col("clin_size_long_diam_mm"),
        F.col("concomitant_biopsy"),
        F.col("dermoscopic_type"),
        F.col("diagnosis_1"),
        F.col("diagnosis_2"),
        F.col("diagnosis_3"),
        F.col("diagnosis_4"),
        F.col("diagnosis_5"),
        F.col("diagnosis_confirm_type"),
        F.col("family_hx_mm"),
        F.col("image_type"),
        F.col("lesion_id"),
        F.col("melanocytic"),
        F.col("patient_id"),
        F.col("personal_hx_mm"),
        F.col("sex"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

try:
    records_written = write_source_metadata(spark, source_metadata_df, bronze_tables["source_metadata"])
    status = "success"
except Exception:
    records_written, status = 0, "failed"
    raise
finally:
    # Logged either way, so a failed ingestion leaves a row too.
    write_ingestion_run(
        spark,
        bronze_tables,
        ingestion_run_id,
        DATASET_CONFIG["dataset_key"],
        DATASET_CONFIG["source_version"],
        started_at,
        datetime.now(timezone.utc),
        records_seen=records_seen,
        records_written=records_written,
        status=status,
    )

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records seen: {records_seen}, written: {records_written}")


## Review

In [ ]:
print_bronze_outputs(spark, dbutils, display, landing_paths, bronze_paths, bronze_tables, DATASET_CONFIG["dataset_key"])
